# Tutorial 5: Perturbation Analysis - Phenotype Switching

In-silico perturbation experiments predict what happens when a TF is knocked down or overexpressed. For the MM_lines melanoma dataset, we'll use this to simulate **phenotype switching** - the transition between melanocytic (MEL) and mesenchymal (MES) states.

## Phenotype Switching in Melanoma

Phenotype switching is a key driver of metastasis and therapy resistance in melanoma.
The SCENIC+ paper showed that knocking down key TFs can induce these transitions:

| Perturbation | Expected Effect |
|--------------|-----------------|  
| **SOX10 KD** | MEL→MES shift (loss of melanocytic identity) |
| **ZEB1 KD** | MES→MEL shift (loss of mesenchymal identity) |

We'll validate this using deepSCENIC's perturbation simulation.

## Setup

In [ ]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load Model and Data

In [ ]:
data_dir = "data/MM_lines_results/"

# Load trained model and data
model = ds.tl.load_model(data_dir + "checkpoints/deepscenic_model.pt", device=device)
mdata = ds.read(data_dir + "preprocessed_data.h5mu")

print(f"Model: {len(model.tf_names)} TFs")

In [ ]:
# Define marker genes for tracking phenotype changes
MEL_MARKERS = ["MITF", "DCT", "TYR", "PMEL"]  # Melanocytic markers
MES_MARKERS = ["JUN", "ZEB1", "NFIB"]   # Mesenchymal markers

print(f"MEL markers: {MEL_MARKERS}")
print(f"MES markers: {MES_MARKERS}")

## SOX10 Knockdown: Proof-of-Principle for MEL→MES Transition

SOX10 is a master regulator of the melanocytic lineage. Knocking it down should cause:
1. **Downregulation of MEL markers**
2. **Upregulation of MES markers**: As cells lose melanocytic identity
3. **Shift toward MES state**: Cells should move toward mesenchymal phenotype

Let's simulate this and verify the expected biology.

In [ ]:
# Simulate SOX10 knockdown with intermediate results for convergence analysis
logFC_trace = ds.tl.simulate_perturbation(
    model,
    mdata,
    tf_name="SOX10",
    level=0,  # Complete knockdown
    n_iter=10,
    return_intermediate=True,  # Get logFC at each iteration
)

# Get final logFC for downstream analysis
logFC_final = logFC_trace[10]

print(f"Simulated SOX10 knockdown across {logFC_final.shape[0]} cells")
print(f"Tracking effects on {logFC_final.shape[1]} genes over 10 iterations")

## Iteration Convergence: How Effects Propagate

The perturbation simulation runs iteratively to capture indirect effects through TF-TF regulatory cascades. Let's visualize how marker gene expression changes across iterations:

In [ ]:
import matplotlib.pyplot as plt

# Get gene indices for markers
gene_names = list(mdata.mod["rna"].var_names)

def get_gene_indices(markers, gene_names):
    """Get indices of marker genes that exist in the dataset."""
    indices = {}
    for m in markers:
        if m in gene_names:
            indices[m] = gene_names.index(m)
    return indices

mel_idx = get_gene_indices(MEL_MARKERS, gene_names)
mes_idx = get_gene_indices(MES_MARKERS, gene_names)

# Extract mean logFC per iteration for each marker
iterations = sorted(logFC_trace.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# MEL markers
for marker, idx in mel_idx.items():
    logfc_per_iter = [logFC_trace[i][:, idx].mean() for i in iterations]
    axes[0].plot(iterations, logfc_per_iter, 'o-', label=marker)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Mean log2FC")
axes[0].set_title("MEL Markers (should decrease)")
axes[0].legend()

# MES markers
for marker, idx in mes_idx.items():
    logfc_per_iter = [logFC_trace[i][:, idx].mean() for i in iterations]
    axes[1].plot(iterations, logfc_per_iter, 'o-', label=marker)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Mean log2FC")
axes[1].set_title("MES Markers (should increase)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Visualize Perturbation Effects

### Waterfall Plot

The waterfall plot shows the top up- and down-regulated genes ranked by effect size:

In [ ]:
# Process results for plotting
results = ds.tl.process_perturbation_results(logFC_final, mdata, tf_name="SOX10")

# Waterfall plot shows top affected genes ranked by effect size
ds.pl.waterfall_perturbation(
    results,
    highlight_genes=MEL_MARKERS + MES_MARKERS,
    title="SOX10 Knockdown Effects"
)

```{tip}
**Expected results for SOX10 knockdown:**

1. **MEL markers down**: MITF, DCT, TYR should have negative log2FC (bars extending left)
2. **MES markers up**: JUN, ZEB1, NFIB should have positive log2FC (bars extending right)
3. **Indirect targets**: Genes regulated by MITF (a SOX10 target) will also change
4. **Self-downregulation**: SOX10 itself should show strong downregulation (confirming the perturbation)

If you see the opposite pattern, check:
- Model training convergence (Tutorial 3)
- GRN quality (Tutorial 4) - does SOX10 have MITF as a target?
```

### Heatmap

Heatmap of top affected genes:

In [ ]:
# Heatmap of top affected genes
ds.pl.heatmap_perturbation(results, top_n=30)

## ZEB1 Knockdown: Reverse Phenotype Switch (MES→MEL)

To validate our GRN, let's simulate the reverse transition. ZEB1 is a mesenchymal TF - knocking it down should cause MES cells to shift toward MEL.

In [ ]:
# Simulate ZEB1 knockdown
logFC_zeb1 = ds.tl.simulate_perturbation(
    model,
    mdata,
    tf_name="ZEB1",
    level=0,
    n_iter=10,
)

results_zeb1 = ds.tl.process_perturbation_results(logFC_zeb1, mdata, tf_name="ZEB1")

# Waterfall plot for ZEB1 KD
ds.pl.waterfall_perturbation(
    results_zeb1,
    highlight_genes=MEL_MARKERS + MES_MARKERS,
    title="ZEB1 Knockdown Effects"
)

### Expected Results for ZEB1 KD

For the reverse transition (MES→MEL):
- **MEL markers should increase** (positive log2FC)
- **MES markers should decrease** (negative log2FC)

This bidirectional validation confirms that the GRN captures the regulatory logic of phenotype switching.

### Dotplot: Comparing SOX10 vs ZEB1 Effects on Marker Genes

A dotplot shows the effect size and direction for specific genes across multiple TF perturbations:

In [ ]:
import pandas as pd

# Combine SOX10 and ZEB1 results for comparison
combined_results = pd.concat([results, results_zeb1])

# Dotplot comparing effects on marker genes
ds.pl.dotplot_perturbation(combined_results, genes=MEL_MARKERS + MES_MARKERS)

## Multi-TF Perturbation

Simulate perturbation of multiple TFs simultaneously:

In [ ]:
# Simulate combined knockdown of two TFs
logFC_multi = ds.tl.simulate_multi_perturbation(
    model,
    mdata,
    tf_names=["SOX10", "MITF"],
    levels=[0, 0],  # Both knocked down
)

results_multi = ds.tl.process_perturbation_results(
    logFC_multi, mdata, tf_name=["SOX10", "MITF"]
)
results_multi.head()

## Comparing TF Effects

Compare perturbation effects across multiple TFs:

In [ ]:
# Heatmap comparing effects of different TF perturbations
# Combine all single-TF and multi-TF results
all_results = pd.concat([results, results_zeb1, results_multi])
ds.pl.heatmap_perturbation(all_results, top_n=30)

## Next Steps

Now that you've analyzed perturbation effects:

- Explore advanced genomic visualizations: {doc}`06_advanced_visualization`